# 03 — v2 Red-Team Audit

Walks through the Phase 10.4 adversarial evaluation pipeline:

- Loading the **110-prompt v2 corpus** (`data/redteam/redteam_corpus_v2.json`)
- Running it against `detect_phi`
- Interpreting the per-category and per-severity pass-rate breakdowns
- Demonstrating that a *deliberately leaky* echo-target trips the harness

The harness is target-agnostic: any `async (str) -> Pydantic-model | dict` can be plugged in.

In [ ]:
import asyncio
import json
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
CORPUS = REPO_ROOT / 'data' / 'redteam' / 'redteam_corpus_v2.json'

## 1. Inspect the corpus

10 attack categories × ~ 11 prompts each. Prompts have:
- `expected_outcome`: one of `{abstain, refuse, execute_safely, redact}`
- `forbidden_substrings`: substrings whose presence in plain output indicates the attack succeeded
- `severity`: one of `{informational, low, moderate, high, critical}`

In [ ]:
from a2a_agent.redteam_v2 import load_corpus

prompts = load_corpus(CORPUS)
print(f"Total prompts: {len(prompts)}")
by_cat: dict[str, int] = {}
for p in prompts:
    by_cat[p.category] = by_cat.get(p.category, 0) + 1
for cat, n in sorted(by_cat.items()):
    print(f"  {cat:<25} {n}")

## 2. Run against `detect_phi`

The PHI scrubber is a strong default target: it surfaces detected PHI inside a structured `redaction_map`, which the harness recognises as a successful redaction (vs an in-prose leak).

In [ ]:
from a2a_agent.redteam_v2 import run_redteam_corpus
from mcp_server.tools.detect_phi import detect_phi

async def phi_target(payload: str):
    return await detect_phi(text=payload)

rep = asyncio.run(run_redteam_corpus(
    CORPUS, phi_target,
    target_label='detect_phi', corpus_id='redteam-v2-2026-04',
))
print(f"Overall pass rate : {rep.overall_pass_rate:.3f} "
      f"({rep.n_passed}/{rep.n_cases})")
print(f"Posture           : {rep.posture}")
print(f"Pass rate by category:")
for cat, rate in sorted(rep.pass_rate_by_category.items()):
    print(f"  {cat:<25} {rate:.3f}")

## 3. Stress-test: a deliberately leaky target

To prove the harness actually catches leaks, we plug in an `echo` target that returns the payload verbatim. PHI cases with explicit forbidden substrings (SSN, MRN, phone) are now expected to **fail**, and the overall posture must flip to `fail`.

In [ ]:
async def echo_target(payload: str):
    return {'echo': payload}

leaky = asyncio.run(run_redteam_corpus(
    CORPUS, echo_target, target_label='echo-target',
))
print(f"Overall pass rate : {leaky.overall_pass_rate:.3f} "
      f"({leaky.n_passed}/{leaky.n_cases})")
print(f"Posture           : {leaky.posture}")
print()
fails = [c for c in leaky.cases if not c.passed][:8]
for c in fails:
    hits = ', '.join(c.forbidden_hits)
    print(f"  {c.prompt_id:<8} {c.category:<22} {c.severity:<10} hits=[{hits}]")

## 4. Generated artefacts

Run the canned driver to regenerate the markdown report + structured JSON in `docs/adversarial/`:

```bash
PYTHONPATH=src .venv/Scripts/python.exe scripts/run_redteam_v2.py
```

Outputs:
- `docs/adversarial/RED_TEAM_RESULTS.md` — human-readable summary
- `docs/adversarial/red_team_run.json` — full `RedTeamReport` payload

Re-run on every CI build to lock the system's adversarial robustness profile against drift.